In [2]:
# ============================================================
# Qwen3-8B — Clinical entity attribute classifier (Vietnamese)
# Optimized for Google Colab T4 GPU (16GB VRAM)
# Labels: isFamily, isHistorical, isNegated, isUncertain
# ============================================================

!pip install -q -U transformers accelerate bitsandbytes

import torch, json, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Sanity check: confirm T4 is attached ----
assert torch.cuda.is_available(), "No GPU found — go to Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen3-8B"  # fallback: "Qwen/Qwen2.5-7B-Instruct" if unavailable

# T4 = Turing architecture -> no native bfloat16 support, use float16 instead
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",  # safer than sdpa/flash-attn on T4
)
model.eval()

SYSTEM_PROMPT = """Bạn là một hệ thống trích xuất thông tin y khoa (clinical NLP) chuyên gán nhãn thuộc tính cho một thực thể y khoa (entity) xuất hiện trong câu bệnh án tiếng Việt.

Bạn sẽ nhận một câu, trong đó thực thể cần phân loại được đánh dấu bằng [ENTITY].

Hãy gán 4 nhãn boolean (true/false) cho thực thể đó, dựa CHỈ trên ngữ cảnh của câu:

1. isFamily: true nếu thực thể đề cập đến tình trạng của NGƯỜI THÂN (cha, mẹ, anh chị em, con...) chứ không phải của chính bệnh nhân. Nếu không có dấu hiệu nào cho thấy đây là người thân, mặc định là false (giả định là bệnh nhân).

2. isHistorical: true nếu thực thể mô tả một tình trạng/sự kiện TRONG QUÁ KHỨ (tiền sử bệnh, đã từng xảy ra, đã điều trị khỏi...) chứ không phải tình trạng hiện tại/lý do nhập viện hiện tại. Nếu là lý do nhập viện hiện tại hoặc triệu chứng hiện tại, gán false.

3. isNegated: true nếu câu PHỦ ĐỊNH sự tồn tại của thực thể (ví dụ: "không có", "không kèm theo", "phủ nhận", "chưa từng"). Nếu thực thể được khẳng định là có xảy ra, gán false.

4. isUncertain: true nếu thực thể được diễn đạt với sự KHÔNG CHẮC CHẮN (ví dụ: "nghi ngờ", "có thể", "chưa rõ", "không đặc hiệu", "theo dõi"). Nếu được khẳng định chắc chắn, gán false.

QUY TẮC QUAN TRỌNG:
- Chỉ dựa vào bằng chứng tường minh trong câu. Không suy diễn thêm.
- Cụm "không đặc hiệu" (unspecified/nonspecific) áp dụng cho isUncertain=true, vì nó thể hiện chẩn đoán chưa rõ ràng/chưa xác định cụ thể — đây KHÔNG phải là phủ định (isNegated vẫn false).
- Trả lời CHỈ bằng một object JSON hợp lệ, không thêm giải thích, không thêm markdown, đúng định dạng:
{"isFamily": bool, "isHistorical": bool, "isNegated": bool, "isUncertain": bool}
"""

FEWSHOT = [
    {
        "role": "user",
        "content": 'Câu: "Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu."\nThực thể: "vết thương phần mềm"'
    },
    {
        "role": "assistant",
        "content": '{"isFamily": false, "isHistorical": false, "isNegated": false, "isUncertain": true}'
    },
    {
        "role": "user",
        "content": 'Câu: "Tiền sử: bố có [ENTITY]."\nThực thể: "tăng huyết áp"'
    },
    {
        "role": "assistant",
        "content": '{"isFamily": true, "isHistorical": true, "isNegated": false, "isUncertain": false}'
    },
    {
        "role": "user",
        "content": 'Câu: "Bệnh nhân không có [ENTITY] kèm theo."\nThực thể: "sốt"'
    },
    {
        "role": "assistant",
        "content": '{"isFamily": false, "isHistorical": false, "isNegated": true, "isUncertain": false}'
    },
]

def classify_entity(sentence: str, entity_text: str) -> dict:
    user_msg = f'Câu: "{sentence}"\nThực thể: "{entity_text}"'

    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + FEWSHOT + \
               [{"role": "user", "content": user_msg}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,  # Qwen3: skip chain-of-thought, go straight to JSON
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(gen, skip_special_tokens=True).strip()

    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in model output: {raw}")
    return json.loads(match.group(0))


# ---------------- Example run ----------------
sentence = "Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu."
entity_text = "vết thương phần mềm"  # replace with your actual extracted entity span

result = classify_entity(sentence, entity_text)
print(json.dumps(result, ensure_ascii=False, indent=2))

print(f"\nGPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
GPU: Tesla T4


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (6 outputs left out)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{
  "isFamily": false,
  "isHistorical": false,
  "isNegated": false,
  "isUncertain": true
}

GPU memory used: 6.10 GB


In [3]:
# ============================================================
# Additional test sentences — covers all label combinations
# ============================================================

test_cases = [
    # 1. Baseline: current condition, no negation/uncertainty/family/history
    {
        "sentence": "Lý do nhập viện: [ENTITY] vùng bụng dưới, mức độ trung bình.",
        "entity": "đau",
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": False}
    },
    # 2. Uncertain only (không đặc hiệu pattern, same as your original)
    {
        "sentence": "Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu.",
        "entity": "vết thương phần mềm",
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True}
    },
    # 3. Negated only
    {
        "sentence": "Bệnh nhân phủ nhận có [ENTITY] trong 24 giờ qua.",
        "entity": "nôn ói",
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": True, "isUncertain": False}
    },
    # 4. Historical only
    {
        "sentence": "Tiền sử bản thân: đã từng phẫu thuật [ENTITY] cách đây 3 năm.",
        "entity": "ruột thừa",
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False}
    },
    # 5. Family only
    {
        "sentence": "Mẹ bệnh nhân hiện đang điều trị [ENTITY] tại bệnh viện khác.",
        "entity": "ung thư vú",
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False}
    },
    # 6. Family + Historical
    {
        "sentence": "Tiền sử gia đình: anh trai từng mắc [ENTITY] năm 2015.",
        "entity": "lao phổi",
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": False, "isUncertain": False}
    },
    # 7. Negated + Historical
    {
        "sentence": "Tiền sử: không ghi nhận [ENTITY] trước đây.",
        "entity": "dị ứng thuốc",
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": True, "isUncertain": False}
    },
    # 8. Uncertain + Historical
    {
        "sentence": "Tiền sử nghi có [ENTITY] cách đây khoảng 2 năm, chưa rõ chẩn đoán chính thức.",
        "entity": "viêm gan B",
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": True}
    },
    # 9. Family + Negated
    {
        "sentence": "Không ai trong gia đình được ghi nhận mắc [ENTITY].",
        "entity": "đái tháo đường",
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": True, "isUncertain": False}
    },
    # 10. Family + Uncertain
    {
        "sentence": "Bố nghi ngờ có [ENTITY] nhưng chưa đi khám xác nhận.",
        "entity": "tăng huyết áp",
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": True}
    },
    # 11. Triple: Family + Historical + Uncertain
    {
        "sentence": "Tiền sử gia đình: ông nội nghi có [ENTITY] nhưng không có hồ sơ y tế xác nhận.",
        "entity": "đột quỵ",
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": False, "isUncertain": True}
    },
    # 12. Negated + Uncertain (tricky: "chưa rõ có... hay không")
    {
        "sentence": "Chưa rõ bệnh nhân có [ENTITY] hay không, cần theo dõi thêm.",
        "entity": "nhiễm trùng huyết",
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True}
    },
    # 13. Current symptom with mild uncertainty phrasing
    {
        "sentence": "Bệnh nhân nhập viện vì [ENTITY], theo dõi thêm để loại trừ biến chứng.",
        "entity": "sốt cao liên tục",
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True}
    },
    # 14. Clear current, negative for family history (common combined clinical phrasing)
    {
        "sentence": "Không có ai trong gia đình từng bị [ENTITY] trước đây.",
        "entity": "động kinh",
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": True, "isUncertain": False}
    },
    # 15. Plain historical, patient's own, no uncertainty
    {
        "sentence": "Bệnh nhân có tiền sử [ENTITY] đã được chẩn đoán và điều trị ổn định.",
        "entity": "hen phế quản",
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False}
    },
]

# ---------------- Run batch evaluation ----------------
correct = 0
for i, case in enumerate(test_cases, 1):
    pred = classify_entity(case["sentence"], case["entity"])
    is_correct = pred == case["expected"]
    correct += is_correct
    status = "✅" if is_correct else "❌"
    print(f"{status} [{i}] Entity: '{case['entity']}'")
    print(f"    Sentence: {case['sentence']}")
    print(f"    Expected: {case['expected']}")
    print(f"    Predicted: {pred}")
    print()

print(f"Accuracy: {correct}/{len(test_cases)} = {correct/len(test_cases)*100:.1f}%")

✅ [1] Entity: 'đau'
    Sentence: Lý do nhập viện: [ENTITY] vùng bụng dưới, mức độ trung bình.
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': False}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': False}

✅ [2] Entity: 'vết thương phần mềm'
    Sentence: Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu.
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}

❌ [3] Entity: 'nôn ói'
    Sentence: Bệnh nhân phủ nhận có [ENTITY] trong 24 giờ qua.
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': True, 'isUncertain': False}
    Predicted: {'isFamily': False, 'isHistorical': True, 'isNegated': True, 'isUncertain': False}

✅ [4] Entity: 'ruột thừa'
    Sentence: Tiền sử bản thân: đã từng phẫu thuật [ENTITY] cách đây 3 năm.
    Expected: 

In [4]:
# ============================================================
# Qwen3-8B — Attribute classifier với candidate tags từ BGE-M3
# isUncertain chỉ true khi isFamily=isHistorical=isNegated=false
# ============================================================

TAG_DESCRIPTIONS = {
    "isFamily": 'true nếu thực thể thuộc về NGƯỜI THÂN (cha, mẹ, anh chị em...) chứ không phải bản thân bệnh nhân.',
    "isHistorical": 'true nếu thực thể là tình trạng TRONG QUÁ KHỨ / tiền sử, không phải lý do nhập viện hiện tại.',
    "isNegated": 'true nếu câu PHỦ ĐỊNH sự tồn tại của thực thể (không có, phủ nhận, chưa từng...).',
    "isUncertain": 'true nếu thực thể được diễn đạt KHÔNG CHẮC CHẮN (nghi ngờ, có thể, chưa rõ, không đặc hiệu, theo dõi). QUAN TRỌNG: isUncertain CHỈ được true khi cả isFamily, isHistorical, isNegated đều false. Nếu thực thể đã thuộc về người thân, hoặc là tiền sử, hoặc bị phủ định — thì isUncertain PHẢI là false, bất kể có từ ngữ mơ hồ hay không.',
}

def build_system_prompt(candidate_tags):
    """Chỉ đưa vào prompt các tag mà BGE-M3 đã chọn ra."""
    tag_list_str = "\n".join(
        f"- {tag}: {TAG_DESCRIPTIONS[tag]}" for tag in candidate_tags
    )
    json_format = "{" + ", ".join(f'"{t}": bool' for t in candidate_tags) + "}"

    return f"""Bạn là hệ thống trích xuất thông tin y khoa (clinical NLP). Bạn nhận một câu bệnh án tiếng Việt với thực thể được đánh dấu [ENTITY], cùng danh sách các nhãn CẦN XÉT (đã được hệ thống truy hồi chọn lọc sẵn — chỉ xét đúng các nhãn này, không xét nhãn nào khác).

Các nhãn cần xét trong lượt này:
{tag_list_str}

QUY TẮC:
- Chỉ dựa vào bằng chứng tường minh trong câu. Không suy diễn thêm.
- Nếu isUncertain nằm trong danh sách cần xét: nó chỉ true khi thực thể KHÔNG thuộc người thân, KHÔNG phải tiền sử, KHÔNG bị phủ định — tức là các nhãn khác trong danh sách (nếu có) đều false.
- Trả lời CHỈ bằng JSON đúng định dạng, không giải thích, không markdown:
{json_format}
"""

# Few-shot mẫu ứng với từng bộ candidate tag khác nhau
FEWSHOT_POOL = [
    # candidates = [isNegated, isUncertain]
    (["isNegated", "isUncertain"],
     'Câu: "Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu."\nThực thể: "vết thương phần mềm"',
     '{"isNegated": false, "isUncertain": true}'),
    (["isNegated", "isUncertain"],
     'Câu: "Bệnh nhân không có [ENTITY] kèm theo."\nThực thể: "sốt"',
     '{"isNegated": true, "isUncertain": false}'),

    # candidates = [isFamily, isHistorical]
    (["isFamily", "isHistorical"],
     'Câu: "Tiền sử gia đình: anh trai từng mắc [ENTITY] năm 2015."\nThực thể: "lao phổi"',
     '{"isFamily": true, "isHistorical": true}'),
    (["isFamily", "isHistorical"],
     'Câu: "Lý do nhập viện: [ENTITY] vùng bụng dưới."\nThực thể: "đau"',
     '{"isFamily": false, "isHistorical": false}'),

    # candidates = [isFamily, isUncertain] — chỗ dễ sai nhất
    (["isFamily", "isUncertain"],
     'Câu: "Bố nghi ngờ có [ENTITY] nhưng chưa đi khám xác nhận."\nThực thể: "tăng huyết áp"',
     '{"isFamily": true, "isUncertain": false}'),  # isUncertain=false vì isFamily=true
    (["isFamily", "isUncertain"],
     'Câu: "Bệnh nhân nghi có [ENTITY], cần theo dõi thêm."\nThực thể: "nhiễm trùng huyết"',
     '{"isFamily": false, "isUncertain": true}'),

    # candidates = [isHistorical, isUncertain]
    (["isHistorical", "isUncertain"],
     'Câu: "Tiền sử nghi có [ENTITY] cách đây khoảng 2 năm, chưa rõ chẩn đoán chính thức."\nThực thể: "viêm gan B"',
     '{"isHistorical": true, "isUncertain": false}'),  # false vì isHistorical=true
]

def get_fewshot_for(candidate_tags):
    """Lọc few-shot khớp đúng bộ tag hiện tại; fallback nếu không có match."""
    matches = [f for f in FEWSHOT_POOL if set(f[0]) == set(candidate_tags)]
    if not matches:
        matches = FEWSHOT_POOL[:2]  # fallback generic
    msgs = []
    for _, user_c, assistant_c in matches[:2]:
        msgs.append({"role": "user", "content": user_c})
        msgs.append({"role": "assistant", "content": assistant_c})
    return msgs

def classify_entity_constrained(sentence: str, entity_text: str, candidate_tags: list) -> dict:
    """
    candidate_tags: subset of ["isFamily","isHistorical","isNegated","isUncertain"],
    lấy từ output của BGE-M3 retrieval step.
    """
    system_prompt = build_system_prompt(candidate_tags)
    fewshot = get_fewshot_for(candidate_tags)
    user_msg = f'Câu: "{sentence}"\nThực thể: "{entity_text}"'

    messages = [{"role": "system", "content": system_prompt}] + fewshot + \
               [{"role": "user", "content": user_msg}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=60, do_sample=False,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(gen, skip_special_tokens=True).strip()

    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in model output: {raw}")
    result = json.loads(match.group(0))

    # Fill missing tags as false (không nằm trong candidate = mặc định false)
    full_result = {t: result.get(t, False) for t in
                    ["isFamily", "isHistorical", "isNegated", "isUncertain"]}
    return full_result


# ============================================================
# Test cases: mỗi câu chỉ đưa 2-3 candidate tags (giả lập BGE-M3)
# ============================================================
constrained_cases = [
    {
        "sentence": "Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu.",
        "entity": "vết thương phần mềm",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Bệnh nhân phủ nhận có [ENTITY] trong 24 giờ qua.",
        "entity": "nôn ói",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Mẹ bệnh nhân hiện đang điều trị [ENTITY] tại bệnh viện khác.",
        "entity": "ung thư vú",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bố nghi ngờ có [ENTITY] nhưng chưa đi khám xác nhận.",
        "entity": "tăng huyết áp",
        "candidates": ["isFamily", "isUncertain"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Tiền sử nghi có [ENTITY] cách đây khoảng 2 năm, chưa rõ chẩn đoán chính thức.",
        "entity": "viêm gan B",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Chưa rõ bệnh nhân có [ENTITY] hay không, cần theo dõi thêm.",
        "entity": "nhiễm trùng huyết",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Không có ai trong gia đình từng bị [ENTITY] trước đây.",
        "entity": "động kinh",
        "candidates": ["isFamily", "isHistorical", "isNegated"],
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân có tiền sử [ENTITY] đã được chẩn đoán và điều trị ổn định.",
        "entity": "hen phế quản",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
]

correct = 0
for i, case in enumerate(constrained_cases, 1):
    pred = classify_entity_constrained(case["sentence"], case["entity"], case["candidates"])
    is_correct = pred == case["expected"]
    correct += is_correct
    status = "✅" if is_correct else "❌"
    print(f"{status} [{i}] Candidates: {case['candidates']}")
    print(f"    Entity: '{case['entity']}'")
    print(f"    Expected: {case['expected']}")
    print(f"    Predicted: {pred}")
    print()

print(f"Accuracy: {correct}/{len(constrained_cases)} = {correct/len(constrained_cases)*100:.1f}%")

✅ [1] Candidates: ['isNegated', 'isUncertain']
    Entity: 'vết thương phần mềm'
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}

✅ [2] Candidates: ['isNegated', 'isUncertain']
    Entity: 'nôn ói'
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': True, 'isUncertain': False}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': True, 'isUncertain': False}

✅ [3] Candidates: ['isFamily', 'isHistorical']
    Entity: 'ung thư vú'
    Expected: {'isFamily': True, 'isHistorical': False, 'isNegated': False, 'isUncertain': False}
    Predicted: {'isFamily': True, 'isHistorical': False, 'isNegated': False, 'isUncertain': False}

✅ [4] Candidates: ['isFamily', 'isUncertain']
    Entity: 'tăng huyết áp'
    Expected: {'isFamily': True, 'isHistorical': False, 'isNegated': False, 'isUncertain': False}
    Predicted

In [5]:
# ============================================================
# Test cases: 30 ví dụ, mỗi câu chỉ đưa 2-3 candidate tags
# (giả lập output từ BGE-M3 retrieval step)
# ============================================================
constrained_cases = [
    # ---------- candidates = [isNegated, isUncertain] ----------
    {
        "sentence": "Lý do nhập viện: [ENTITY] do chấn thương, không đặc hiệu.",
        "entity": "vết thương phần mềm",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Bệnh nhân phủ nhận có [ENTITY] trong 24 giờ qua.",
        "entity": "nôn ói",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Chưa rõ bệnh nhân có [ENTITY] hay không, cần theo dõi thêm.",
        "entity": "nhiễm trùng huyết",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Không ghi nhận [ENTITY] trên bệnh nhân tại thời điểm khám.",
        "entity": "khó thở",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Nghi ngờ [ENTITY], cần làm thêm xét nghiệm để xác định.",
        "entity": "viêm ruột thừa",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Bệnh nhân được chẩn đoán xác định [ENTITY], không có nghi vấn gì thêm.",
        "entity": "viêm phổi thùy",
        "candidates": ["isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },

    # ---------- candidates = [isFamily, isHistorical] ----------
    {
        "sentence": "Mẹ bệnh nhân hiện đang điều trị [ENTITY] tại bệnh viện khác.",
        "entity": "ung thư vú",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Lý do nhập viện: [ENTITY] vùng bụng dưới.",
        "entity": "đau",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Tiền sử gia đình: anh trai từng mắc [ENTITY] năm 2015.",
        "entity": "lao phổi",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân có tiền sử bản thân [ENTITY], đã điều trị dứt điểm cách đây 5 năm.",
        "entity": "sỏi thận",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Con gái bệnh nhân từng được chẩn đoán [ENTITY] lúc nhỏ, hiện đã khỏi.",
        "entity": "hen suyễn",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân nhập viện lần này vì [ENTITY] khởi phát đột ngột.",
        "entity": "đau ngực trái",
        "candidates": ["isFamily", "isHistorical"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },

    # ---------- candidates = [isFamily, isUncertain] ----------
    {
        "sentence": "Bố nghi ngờ có [ENTITY] nhưng chưa đi khám xác nhận.",
        "entity": "tăng huyết áp",
        "candidates": ["isFamily", "isUncertain"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân nghi có [ENTITY], cần theo dõi thêm.",
        "entity": "nhiễm trùng huyết",
        "candidates": ["isFamily", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Chị gái bệnh nhân được chẩn đoán chắc chắn mắc [ENTITY].",
        "entity": "lupus ban đỏ",
        "candidates": ["isFamily", "isUncertain"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Có thể bệnh nhân bị [ENTITY], chưa loại trừ hoàn toàn.",
        "entity": "sốt xuất huyết",
        "candidates": ["isFamily", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Ông ngoại nghi có tiền sử [ENTITY] nhưng gia đình không rõ chi tiết.",
        "entity": "đột quỵ",
        "candidates": ["isFamily", "isUncertain"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },

    # ---------- candidates = [isHistorical, isUncertain] ----------
    {
        "sentence": "Tiền sử nghi có [ENTITY] cách đây khoảng 2 năm, chưa rõ chẩn đoán chính thức.",
        "entity": "viêm gan B",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân có tiền sử [ENTITY] đã được chẩn đoán và điều trị ổn định.",
        "entity": "hen phế quản",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Nghi ngờ bệnh nhân đang có đợt cấp của [ENTITY], cần theo dõi thêm triệu chứng.",
        "entity": "viêm loét dạ dày",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },
    {
        "sentence": "Trước đây từng nghi ngờ [ENTITY] nhưng không có hồ sơ xác nhận rõ ràng.",
        "entity": "rối loạn nhịp tim",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Lý do nhập viện hiện tại: theo dõi [ENTITY] chưa xác định nguyên nhân.",
        "entity": "co giật",
        "candidates": ["isHistorical", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": False, "isUncertain": True},
    },

    # ---------- candidates = [isFamily, isNegated] ----------
    {
        "sentence": "Không ai trong gia đình được ghi nhận mắc [ENTITY].",
        "entity": "đái tháo đường",
        "candidates": ["isFamily", "isNegated"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Em trai bệnh nhân hiện đang mắc [ENTITY].",
        "entity": "viêm khớp dạng thấp",
        "candidates": ["isFamily", "isNegated"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân không có [ENTITY] tại thời điểm nhập viện.",
        "entity": "phù chân",
        "candidates": ["isFamily", "isNegated"],
        "expected": {"isFamily": False, "isHistorical": False, "isNegated": True, "isUncertain": False},
    },

    # ---------- candidates = [isHistorical, isNegated] ----------
    {
        "sentence": "Tiền sử: không ghi nhận [ENTITY] trước đây.",
        "entity": "dị ứng thuốc",
        "candidates": ["isHistorical", "isNegated"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân từng được điều trị [ENTITY] cách đây 1 năm, hiện đã ổn định.",
        "entity": "viêm túi mật",
        "candidates": ["isHistorical", "isNegated"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân chưa từng bị [ENTITY] trong suốt quá trình bệnh trước đây.",
        "entity": "xuất huyết tiêu hóa",
        "candidates": ["isHistorical", "isNegated"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": True, "isUncertain": False},
    },

    # ---------- candidates = 3 tags ----------
    {
        "sentence": "Không có ai trong gia đình từng bị [ENTITY] trước đây.",
        "entity": "động kinh",
        "candidates": ["isFamily", "isHistorical", "isNegated"],
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Tiền sử gia đình: ông nội nghi có [ENTITY] nhưng không có hồ sơ y tế xác nhận.",
        "entity": "đột quỵ",
        "candidates": ["isFamily", "isHistorical", "isUncertain"],
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Bệnh nhân phủ nhận tiền sử [ENTITY] trong gia đình.",
        "entity": "bệnh tim mạch",
        "candidates": ["isFamily", "isHistorical", "isNegated"],
        "expected": {"isFamily": True, "isHistorical": True, "isNegated": True, "isUncertain": False},
    },
    {
        "sentence": "Chưa rõ liệu bệnh nhân có từng mắc [ENTITY] trong quá khứ hay không.",
        "entity": "viêm màng não",
        "candidates": ["isHistorical", "isNegated", "isUncertain"],
        "expected": {"isFamily": False, "isHistorical": True, "isNegated": False, "isUncertain": False},
    },
    {
        "sentence": "Dì của bệnh nhân nghi có [ENTITY] nhưng chưa đi khám để xác nhận.",
        "entity": "u xơ tử cung",
        "candidates": ["isFamily", "isNegated", "isUncertain"],
        "expected": {"isFamily": True, "isHistorical": False, "isNegated": False, "isUncertain": False},
    },
]

print(f"Tổng số ví dụ: {len(constrained_cases)}\n")

correct = 0
tag_correct = {"isFamily": 0, "isHistorical": 0, "isNegated": 0, "isUncertain": 0}
tag_total = {"isFamily": 0, "isHistorical": 0, "isNegated": 0, "isUncertain": 0}

for i, case in enumerate(constrained_cases, 1):
    pred = classify_entity_constrained(case["sentence"], case["entity"], case["candidates"])
    is_correct = pred == case["expected"]
    correct += is_correct

    for tag in case["candidates"]:
        tag_total[tag] += 1
        if pred[tag] == case["expected"][tag]:
            tag_correct[tag] += 1

    status = "✅" if is_correct else "❌"
    print(f"{status} [{i}] Candidates: {case['candidates']}")
    print(f"    Entity: '{case['entity']}'")
    print(f"    Expected: {case['expected']}")
    print(f"    Predicted: {pred}")
    if not is_correct:
        print(f"    Sentence: {case['sentence']}")
    print()

print(f"{'='*60}")
print(f"Overall Accuracy: {correct}/{len(constrained_cases)} = {correct/len(constrained_cases)*100:.1f}%")
print(f"\nPer-tag accuracy (chỉ tính khi tag nằm trong candidates):")
for tag in tag_total:
    if tag_total[tag] > 0:
        print(f"  {tag}: {tag_correct[tag]}/{tag_total[tag]} = {tag_correct[tag]/tag_total[tag]*100:.1f}%")

Tổng số ví dụ: 33

✅ [1] Candidates: ['isNegated', 'isUncertain']
    Entity: 'vết thương phần mềm'
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}

✅ [2] Candidates: ['isNegated', 'isUncertain']
    Entity: 'nôn ói'
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': True, 'isUncertain': False}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': True, 'isUncertain': False}

✅ [3] Candidates: ['isNegated', 'isUncertain']
    Entity: 'nhiễm trùng huyết'
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}
    Predicted: {'isFamily': False, 'isHistorical': False, 'isNegated': False, 'isUncertain': True}

✅ [4] Candidates: ['isNegated', 'isUncertain']
    Entity: 'khó thở'
    Expected: {'isFamily': False, 'isHistorical': False, 'isNegated': True, 'isUncertain':